# Übungen – Block 4: Ansible – Installation, Setup und erste Automatisierung

Die Übungen benötigen einen vorbereiteten Control Node sowie mindestens einen erreichbaren Linux-Managed-Node.

## Musterlösung

Hostnamen, Benutzer und Paketnamen müssen an die Schulungsumgebung angepasst werden.

## Übung 1: Installation prüfen

1. Prüfen Sie die installierte Ansible-Version.
2. Ermitteln Sie aus der Ausgabe, welche Python-Version und welche Konfigurationsdatei verwendet werden.
3. Notieren Sie den Pfad zur ausführbaren Ansible-Installation.

```bash
ansible --version
which ansible
```

## Übung 2: Projektverzeichnis und Inventory

Erstellen Sie ein Projektverzeichnis `~/ansible-training/block4`.

Legen Sie ein Inventory mit mindestens zwei Gruppen an, zum Beispiel `web` und `db`. Prüfen Sie anschließend das Inventory mit `ansible-inventory`.

```bash
mkdir -p ~/ansible-training/block4
cd ~/ansible-training/block4
```

Beispiel `inventory.ini`:

```ini
[web]
web01
web02

[db]
db01
```

Prüfung:

```bash
ansible-inventory -i inventory.ini --graph
```

## Übung 3: Verbindung testen

1. Testen Sie die direkte SSH-Verbindung zu einem Managed Node.
2. Testen Sie anschließend alle Inventory-Hosts mit dem Ansible-Modul `ping`.
3. Untersuchen Sie mögliche Unterschiede zwischen `UNREACHABLE` und `FAILED`.

```bash
ssh training@web01
ansible all -i inventory.ini -m ping
```

`UNREACHABLE` weist typischerweise auf ein Verbindungsproblem hin. `FAILED` bedeutet, dass der Host erreicht wurde, die eigentliche Aufgabe aber fehlschlug.

## Übung 4: Ad-hoc Commands

Führen Sie auf einer Hostgruppe folgende Aufgaben aus:

- Hostnamen ausgeben,
- aktuelle Laufzeit anzeigen,
- eine kleine Textdatei nach `/tmp` kopieren,
- das Vorhandensein der Datei prüfen.

```bash
ansible web -i inventory.ini -m command -a "hostname"
ansible web -i inventory.ini -m command -a "uptime"

echo "Block 4" > demo.txt
ansible web -i inventory.ini -m copy -a "src=demo.txt dest=/tmp/demo.txt"
ansible web -i inventory.ini -m command -a "cat /tmp/demo.txt"
```

## Übung 5: Paketverwaltung und become

Installieren Sie ein vom Trainer freigegebenes kleines Paket auf einer Hostgruppe.

1. Führen Sie den Task zunächst ohne `become` aus.
2. Interpretieren Sie das Ergebnis.
3. Wiederholen Sie den Task mit `become`.
4. Führen Sie denselben Befehl erneut aus und beobachten Sie die Ausgabe.

```bash
ansible web -i inventory.ini -m package -a "name=tree state=present"
ansible web -i inventory.ini -b -m package -a "name=tree state=present"
ansible web -i inventory.ini -b -m package -a "name=tree state=present"
```

Beim zweiten erfolgreichen Lauf sollte im Idealfall keine erneute Änderung notwendig sein.

## Abschlussübung

Erstellen Sie einen kleinen Administrationsdurchlauf ausschließlich mit Ad-hoc Commands:

1. Connectivity Check
2. Hostname und Uptime ermitteln
3. Testdatei verteilen
4. Paket installieren
5. Ergebnis pro Host kontrollieren

Dokumentieren Sie jeweils Modul, Zielgruppe und Ergebnis.

```bash
ansible all -i inventory.ini -m ping
ansible all -i inventory.ini -m command -a "hostname"
ansible all -i inventory.ini -m command -a "uptime"
ansible web -i inventory.ini -m copy -a "src=demo.txt dest=/tmp/demo.txt"
ansible web -i inventory.ini -b -m package -a "name=tree state=present"
```

## Erläuterungen und Antworten zu allen Teilfragen

### Übung 1 – Ausgabe von `ansible --version`
Die Ausgabe nennt unter anderem `ansible [core ...]`, den Konfigurationsdateipfad (`config file`), die verwendete Python-Version und den Python-Pfad. `command -v ansible` ermittelt den tatsächlich über `PATH` gefundenen ausführbaren Befehl; dieser kann von der Python-Installation abweichen.

### Übung 2 – Inventory validieren
`ansible-inventory -i inventory.ini --graph` prüft die Gruppenstruktur; `--list` zeigt die aufgelösten Inventory-Daten. Ein korrekt geparstes Inventory garantiert noch keine Erreichbarkeit. Hostnamen, SSH-Benutzer und Schlüssel müssen zur Trainingsumgebung passen.

### Übung 3 – `UNREACHABLE` und `FAILED`
`UNREACHABLE` bedeutet, dass Ansible keine nutzbare Verbindung zum Ziel herstellen konnte, etwa wegen DNS, SSH, Firewall oder Authentifizierung. `FAILED` bezeichnet eine fehlgeschlagene Task-Ausführung nach dem Verbindungsaufbau, etwa durch fehlende Rechte, fehlendes Python oder ungültige Modulparameter. Zur Diagnose zuerst `ssh <benutzer>@<host>`, danach `ansible ... -m ansible.builtin.ping -vvv` verwenden. Das Ansible-`ping`-Modul prüft die Modul-Ausführung und ist kein ICMP-Ping.

### Übung 4 – Module und Zielzustand
`command` führt Programme ohne Shell-Interpretation aus; `copy` sorgt für den gewünschten Dateiinhalt. Zum Prüfen der übertragenen Datei ist `stat` oder `command -a 'cat /tmp/demo.txt'` geeignet. Eine wiederholte `copy`-Ausführung meldet bei unverändertem Inhalt normalerweise `ok`, nicht `changed`.

### Übung 5 – Rechteerhöhung und Idempotenz
Ohne `become` kann die Paketinstallation wegen fehlender Berechtigung scheitern; sie kann aber auch erfolgreich sein, etwa wenn das Paket bereits installiert ist. `-b` aktiviert die Rechteerhöhung. Nach erfolgreicher Installation sollte die erneute Ausführung mit `state=present` keine Änderung melden. `changed=0` setzt einen bereits erreichten Zielzustand voraus und ist kein universelles Erfolgskriterium.

### Abschlussübung – Dokumentation pro Host
Erfassen Sie für jeden Schritt Zielgruppe, Modul, Ergebnisstatus und gegebenenfalls Fehlerursache. Ein `ping`-Erfolg belegt nur die grundsätzliche Ausführbarkeit; Paketinstallation und Dateiverteilung sind separat zu prüfen. Bei einem Fehlschlag darf die Gesamtauswertung nicht stillschweigend nur die erfolgreichen Hosts berücksichtigen.